# Pretrain Train/Val 분할 결과 확인

`data/processed/pretrain_split/` 아래의 압축 JSONL 샤드를 읽어 Train/Val 구조, 샤드 크기, 샘플 문서를 확인한다.

전체 문서 수를 세는 셀은 1,748만 문서를 모두 읽으므로 시간이 걸릴 수 있다.

In [ ]:
from pathlib import Path
import gzip
import json
import time
from itertools import islice

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = ROOT / 'data' / 'processed' / 'pretrain_split'
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'

assert DATA_ROOT.exists(), f'결과 폴더가 없습니다: {DATA_ROOT}'
print(f'결과 폴더: {DATA_ROOT.resolve()}')

In [ ]:
# Train/Val 샤드 목록과 압축 파일 크기를 확인한다.
def shard_summary(folder):
    rows = []
    for path in sorted(folder.glob('*.jsonl.gz')):
        rows.append({
            'split': folder.name,
            'file': path.name,
            'size_gb': round(path.stat().st_size / 1024**3, 3),
        })
    return rows

files = shard_summary(TRAIN_DIR) + shard_summary(VAL_DIR)
for row in files:
    print(f"{row['split']:>5}  {row['file']}  {row['size_gb']:.3f} GB")
print(f'샤드 수: {len(files)}개')

In [ ]:
# 각 분할에서 원하는 개수만 메모리에 읽어 샘플을 확인한다.
def read_samples(split='train', shard='00000.jsonl.gz', count=3):
    folder = DATA_ROOT / split
    path = folder / shard
    if not path.exists():
        raise FileNotFoundError(path)
    with gzip.open(path, 'rt', encoding='utf-8') as file:
        return [json.loads(line) for line in islice(file, count)]

samples = read_samples('train', count=3)
for index, sample in enumerate(samples, start=1):
    print(f'--- Train 샘플 {index} ---')
    print('id:', sample.get('id'))
    print('metadata:', sample.get('metadata'))
    print('text:', sample.get('text', '')[:500])


In [ ]:
# Val 샘플도 같은 방식으로 확인한다.
val_samples = read_samples('val', count=3)
for index, sample in enumerate(val_samples, start=1):
    print(f'--- Val 샘플 {index} ---')
    print('id:', sample.get('id'))
    print('text:', sample.get('text', '')[:500])


In [ ]:
# 필요할 때만 실행: 전체 문서 수와 실제 분할 비율을 계산한다.
def count_jsonl_documents(folder):
    total = 0
    for path in sorted(folder.glob('*.jsonl.gz')):
        started = time.perf_counter()
        with gzip.open(path, 'rt', encoding='utf-8') as file:
            count = sum(1 for _ in file)
        total += count
        print(f'{folder.name}/{path.name}: {count:,}개 ({time.perf_counter() - started:.1f}초)')
    return total

started = time.perf_counter()
train_count = count_jsonl_documents(TRAIN_DIR)
val_count = count_jsonl_documents(VAL_DIR)
total_count = train_count + val_count
print(f'전체: {total_count:,}개')
print(f'Train: {train_count:,}개 ({train_count / total_count:.4%})')
print(f'Val: {val_count:,}개 ({val_count / total_count:.4%})')
print(f'문서 수 계산 시간: {time.perf_counter() - started:.1f}초')